# Matching docente ↔ convocatoria (José)

Piloto de **embeddings (OpenRouter) + TF-IDF**.

- Decisiones: [matching/DECISIONES.md](matching/DECISIONES.md)
- Código: `matching/run_match.py`
- Salidas: `salidas/matching/`


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PERSONA = "jose"
NB_DIR = Path.cwd()
LAB_DIR = NB_DIR.parent if NB_DIR.name != "jose" else NB_DIR.parent
# notebook puede abrirse desde jose/ o desde matching/
JOSE = NB_DIR if (NB_DIR / "matching").exists() else NB_DIR.parent
if JOSE.name != "jose":
    JOSE = NB_DIR
LAB = JOSE.parent
CONVOCAUR = LAB.parent

sys.path.insert(0, str(CONVOCAUR / "src"))
sys.path.insert(0, str(LAB / "_comun"))
sys.path.insert(0, str(JOSE / "matching"))

from cargar_datos import cargar_todo

datos = cargar_todo("jose")
OUT = JOSE / "salidas" / "matching"
print("Salidas matching:", OUT)
print("Archivos:", sorted(p.name for p in OUT.glob("ranking_*.csv")) if OUT.exists() else "(correr run_match.py primero)")


## Paso a paso (resumen)

1. Armar texto de la convocatoria (NLP).
2. Armar texto de cada docente (HUB + CvLAC).
3. Baseline TF-IDF + cosine.
4. Embeddings OpenRouter + cosine.
5. Score híbrido `0.7*emb + 0.3*tfidf` + boost categoría.
6. Top-k → CSV en `salidas/matching/`.


In [ ]:
# Ver rankings ya generados
for conv in ["45", "48", "976"]:
    path = OUT / f"ranking_convocatoria_{conv}.csv"
    if not path.exists():
        print("Falta", path.name, "→ ejecuta: python matching/run_match.py")
        continue
    df = pd.read_csv(path)
    print("\n===", conv, "===")
    print(df.head(5)[["rank", "nombre", "facultad", "categoria", "score_final", "score_emb", "score_tfidf"]].to_string(index=False))


In [ ]:
# Re-correr ranking rápido (TF-IDF only) desde el notebook
# Descomenta para ejecutar sin API:
# import subprocess, sys
# subprocess.check_call([sys.executable, str(JOSE / "matching" / "run_match.py"), "--sin-embeddings", "--top", "10"])
pass


## Zona libre — análisis del match
